# 02.1 Optimized Random Forest Classification

## Objective
This notebook performs a notebook-friendly random forest tuning pass on the processed student dropout dataset. It uses a small validation split and a focused set of strong candidate configurations so it finishes much faster than a large grid search, then saves the chosen model, metrics, confusion matrix, and feature importances into `results/`.

## 1. Import Libraries and Utilities

We import the libraries and shared utility functions needed for loading data, evaluating candidate models, and saving the optimized random forest artifacts.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import (
    METRICS_DIR,
    MODELS_DIR,
    TARGET_LABEL_ORDER,
    ensure_directories,
    evaluate_predictions,
    load_processed_data,
    save_confusion_matrix,
)

RANDOM_STATE = 42
MODEL_NAME = "Optimized Random Forest"
ensure_directories()

## 2. Load Data and Inspect Class Balance

We load the processed train-test split and inspect the dataset size and target distribution before running the optimization sweep.

In [2]:
X_train, X_test, y_train, y_test = load_processed_data()

dataset_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(X_train), len(X_test)],
        "columns": [X_train.shape[1], X_test.shape[1]],
    }
)
class_balance = pd.concat(
    [
        y_train.value_counts().rename("train_count"),
        y_test.value_counts().rename("test_count"),
    ],
    axis=1,
).reindex(TARGET_LABEL_ORDER)

display(dataset_summary)
display(class_balance)

,split,rows,columns
0,train,3539,243
1,test,885,243


,train_count,test_count
target,,
Dropout,1137,284
Enrolled,635,159
Graduate,1767,442


## 3. Run Focused Hyperparameter Sweep

Instead of a large grid search, this block evaluates a small set of strong random forest candidates on a validation split and selects the best one using validation performance.

In [3]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

candidate_params = [
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 2, "criterion": "gini", "max_features": "sqrt", "class_weight": "balanced", "bootstrap": True, "ccp_alpha": 0.0},
    {"n_estimators": 400, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 2, "criterion": "gini", "max_features": "sqrt", "class_weight": "balanced", "bootstrap": True, "ccp_alpha": 0.0},
    {"n_estimators": 250, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1, "criterion": "gini", "max_features": "sqrt", "class_weight": "balanced", "bootstrap": True, "ccp_alpha": 0.0},
    {"n_estimators": 300, "max_depth": 24, "min_samples_split": 2, "min_samples_leaf": 2, "criterion": "gini", "max_features": "sqrt", "class_weight": "balanced", "bootstrap": True, "ccp_alpha": 0.0},
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 4, "min_samples_leaf": 2, "criterion": "gini", "max_features": "sqrt", "class_weight": "balanced", "bootstrap": True, "ccp_alpha": 0.0},
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 2, "criterion": "log_loss", "max_features": "sqrt", "class_weight": "balanced", "bootstrap": True, "ccp_alpha": 0.0},
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 2, "criterion": "gini", "max_features": 0.5, "class_weight": "balanced", "bootstrap": True, "ccp_alpha": 0.0},
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 2, "criterion": "gini", "max_features": "sqrt", "class_weight": "balanced_subsample", "bootstrap": True, "ccp_alpha": 0.0},
]

search_rows = []
for idx, params in enumerate(candidate_params, start=1):
    model = RandomForestClassifier(
        **params,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    model.fit(X_fit, y_fit)
    val_pred = model.predict(X_val)
    search_rows.append(
        {
            "candidate": idx,
            **params,
            "val_accuracy": float((val_pred == y_val).mean()),
            "val_balanced_accuracy": float(balanced_accuracy_score(y_val, val_pred)),
            "val_f1_weighted": float(f1_score(y_val, val_pred, average="weighted", zero_division=0)),
            "val_f1_macro": float(f1_score(y_val, val_pred, average="macro", zero_division=0)),
        }
    )

search_results_df = (
    pd.DataFrame(search_rows)
    .sort_values(["val_f1_weighted", "val_f1_macro", "val_balanced_accuracy", "val_accuracy"], ascending=False)
    .reset_index(drop=True)
)
best_params = candidate_params[int(search_results_df.loc[0, "candidate"]) - 1]
best_model = RandomForestClassifier(
    **best_params,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
best_model.fit(X_train, y_train)

display(search_results_df[["candidate", "val_accuracy", "val_balanced_accuracy", "val_f1_weighted", "val_f1_macro", "n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "criterion", "max_features", "class_weight"]])
pd.DataFrame([best_params])

,candidate,val_accuracy,val_balanced_accuracy,val_f1_weighted,val_f1_macro,n_estimators,max_depth,min_samples_split,min_samples_leaf,criterion,max_features,class_weight
0,2,0.775424,0.704972,0.768592,0.711999,400,NaN,2,2,gini,sqrt,balanced
1,4,0.774011,0.702874,0.767136,0.709055,300,24.0,2,2,gini,sqrt,balanced
2,1,0.772599,0.702562,0.766005,0.709284,300,NaN,2,2,gini,sqrt,balanced
3,5,0.772599,0.702562,0.766005,0.709284,300,NaN,4,2,gini,sqrt,balanced
4,8,0.769774,0.696786,0.762618,0.702996,300,NaN,2,2,gini,sqrt,balanced_subsample
5,6,0.766949,0.691434,0.758820,0.697286,300,NaN,2,2,log_loss,sqrt,balanced
6,7,0.768362,0.686478,0.758809,0.694458,300,NaN,2,2,gini,0.5,balanced
7,3,0.772599,0.675633,0.754039,0.682977,250,NaN,2,1,gini,sqrt,balanced


,n_estimators,max_depth,min_samples_split,min_samples_leaf,criterion,max_features,class_weight,bootstrap,ccp_alpha
0,400,None,2,2,gini,sqrt,balanced,True,0.0


## 4. Evaluate and Save the Optimized Model

After selecting the best candidate, we evaluate it on the test set and save the model and metrics files for the optimized random forest.

In [4]:
y_pred = best_model.predict(X_test)
train_pred = best_model.predict(X_train)

metrics = evaluate_predictions(y_test, y_pred)
metrics["Model"] = MODEL_NAME
metrics["Best Parameters"] = best_params
metrics["Search Strategy"] = "Focused validation sweep over 8 notebook-friendly candidates"
metrics["Validation Accuracy"] = float(search_results_df.loc[0, "val_accuracy"])
metrics["Validation Balanced Accuracy"] = float(search_results_df.loc[0, "val_balanced_accuracy"])
metrics["Validation Weighted F1"] = float(search_results_df.loc[0, "val_f1_weighted"])
metrics["Validation Macro F1"] = float(search_results_df.loc[0, "val_f1_macro"])
metrics["Training Accuracy"] = float(best_model.score(X_train, y_train))
metrics["Training Macro F1"] = float(f1_score(y_train, train_pred, average="macro", zero_division=0))
metrics["Training Balanced Accuracy"] = float(balanced_accuracy_score(y_train, train_pred))
metrics["CV Accuracy Mean"] = None
metrics["CV Accuracy Std"] = None
metrics["CV Balanced Accuracy Mean"] = None
metrics["CV Weighted F1 Mean"] = None
metrics["CV Macro F1 Mean"] = None

confusion_path = save_confusion_matrix(MODEL_NAME, y_test, y_pred)
metrics["Confusion Matrix Figure"] = str(confusion_path)

feature_importance_df = (
    pd.DataFrame({"feature": X_train.columns, "importance": best_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance_path = METRICS_DIR / "optimized_random_forest_feature_importance.csv"
feature_importance_df.to_csv(feature_importance_path, index=False)
metrics["Feature Importance File"] = str(feature_importance_path)
metrics["Top 10 Features"] = feature_importance_df.head(10).to_dict(orient="records")

metrics_path = METRICS_DIR / "optimized_random_forest.json"
model_path = MODELS_DIR / "optimized_random_forest.joblib"
summary_path = METRICS_DIR / "optimized_random_forest_summary.csv"

metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
joblib.dump(best_model, model_path)
pd.DataFrame(
    [
        {
            "Algorithm": MODEL_NAME,
            "Training Accuracy": metrics["Training Accuracy"],
            "Accuracy (Test)": metrics["Accuracy"],
            "Balanced Accuracy": metrics["Balanced Accuracy"],
            "Weighted F1": metrics["F1-score (weighted)"],
            "Macro F1": metrics["F1-score (macro)"],
            "Validation Weighted F1": metrics["Validation Weighted F1"],
            "Validation Macro F1": metrics["Validation Macro F1"],
        }
    ]
).to_csv(summary_path, index=False)

display(pd.DataFrame([metrics["Best Parameters"]]))
display(pd.DataFrame([{k: v for k, v in metrics.items() if isinstance(v, (int, float, str)) and k not in ["Model", "Confusion Matrix Figure", "Feature Importance File"]}]).transpose())

,n_estimators,max_depth,min_samples_split,min_samples_leaf,criterion,max_features,class_weight,bootstrap,ccp_alpha
0,400,None,2,2,gini,sqrt,balanced,True,0.0


,0
Accuracy,0.771751
Balanced Accuracy,0.710238
Precision (weighted),0.770415
Recall (weighted),0.771751
F1-score (weighted),0.769472
F1-score (macro),0.715083
Search Strategy,Focused validation sweep over 8 notebook-frien...
Validation Accuracy,0.775424
Validation Balanced Accuracy,0.704972
Validation Weighted F1,0.768592


## 5. Review Feature Importances

Finally, we inspect the most influential encoded features identified by the optimized random forest model.

In [5]:
feature_importance_df.head(15)

,feature,importance
0,num__curricular_units_2nd_sem_(approved),0.105644
1,num__curricular_units_2nd_sem_(grade),0.088041
2,num__curricular_units_1st_sem_(approved),0.077468
3,num__curricular_units_1st_sem_(grade),0.065931
4,num__curricular_units_2nd_sem_(evaluations),0.043545
5,num__curricular_units_1st_sem_(evaluations),0.040941
6,num__admission_grade,0.033964
7,num__age_at_enrollment,0.033070
8,num__previous_qualification_(grade),0.030832
9,cat__tuition_fees_up_to_date_0,0.028347
